# Building a ReAct Agent with Memory

In this notebook we'll:

1. Set up a web search tool (Tavily)
2. Build a ReAct agent using `ToolNode` + `tools_condition`
3. Add **memory** with `MemorySaver` for persistent conversations

This builds on the fundamentals from Notebook 1 (State, Nodes, Edges, Tools).

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain-openai langchain-community tavily-python

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")
_set_env("TAVILY_API_KEY")

---
## Step 1: Set Up the Search Tool

We'll use Tavily for web search. Let's verify it works before wiring it into the agent.

In [ ]:
from langchain_community.tools import TavilySearchResults

search_tool = TavilySearchResults(max_results=3)
tools = [search_tool]

# Quick test
search_tool.invoke({"query": "LangGraph latest release"})

## Step 2: Set Up the LLM with Tools

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# Verify tool binding works
output = llm_with_tools.invoke("What is the weather in Tokyo?")
print("Tool calls:", output.tool_calls)

## Step 3: Build the Agent Graph

The ReAct pattern is a loop: **LLM decides** → **Tool executes** → **LLM reflects** → repeat until done.

The assistant node includes a system message to guide the agent's behavior.

In [ ]:
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import tools_condition, ToolNode
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import Image, display

def assistant(state: MessagesState):
    sys_msg = "You are a helpful assistant that can search the web to answer questions. Always cite your sources."
    return {"messages": llm_with_tools.invoke([SystemMessage(sys_msg)] + state["messages"])}

# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
messages = [HumanMessage(content="What are the top 3 trending AI tools right now?")]
output = graph.invoke({"messages": messages})

for m in output["messages"]:
    m.pretty_print()

---
## Step 4: Adding Memory

Right now, each `invoke()` call starts fresh — the agent has no memory of previous conversations. Let's fix that.

<img src="./assets-resources/2025-02-10-16-25-13.png" width=50%>

### Key Concepts

- **Memory**: The ability to process, store, and recall information from past interactions
- **Persistence**: LangGraph saves snapshots of graph state using **checkpointers** at every superstep
- **Threads**: Individual sessions/conversations identified by a unique `thread_id`
- **Checkpoints**: Saved state snapshots that enable resumption, time-travel, and human-in-the-loop

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

# Re-compile the same graph, but now with a checkpointer
graph = builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

### Using Thread IDs

The `thread_id` in the config tells the checkpointer which conversation to load/save. Same thread = same conversation history.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# First message in thread 1
messages = [HumanMessage(content="What is the weather in Tokyo?")]
output = graph.invoke({"messages": messages}, config)

for m in output["messages"]:
    m.pretty_print()

In [ ]:
# Follow-up in the SAME thread — the agent remembers!
new_message = HumanMessage(content="What was the city from the previous question?")
output = graph.invoke({"messages": [new_message]}, config)

for m in output["messages"]:
    m.pretty_print()

In [ ]:
# A DIFFERENT thread — isolated conversation, no memory of thread 1
config2 = {"configurable": {"thread_id": "2"}}

output = graph.invoke(
    {"messages": [HumanMessage(content="What was the city from the previous question?")]},
    config2
)

# The agent won't know — this is a fresh thread
for m in output["messages"]:
    m.pretty_print()

---
## Summary

| What we built | Key API |
|---------------|--------|
| ReAct agent with web search | `ToolNode`, `tools_condition`, `TavilySearchResults` |
| Persistent memory | `MemorySaver`, `thread_id` in config |
| Isolated conversations | Different `thread_id` values |

**Next**: In Notebook 3, we'll build a RAG agent that retrieves from documents and can fall back to web search.